In [2]:
import os,sys
sys.path.insert(1, os.path.join(os.getcwd()  , '..'))

In [3]:
import importlib, shallowsim as sb
import pandas as pd
import math

In [10]:
importlib.reload(sb)

<module 'shallowsim' from 'c:\\Users\\陈铂英\\Desktop\\shallowsim\\shallowsim.py'>

In [12]:
args = sb.ModelArgs.load_from_csv("modelArgs.csv","deepseek-v3")

In [17]:
print(args.is_moe)
print(args.attention)

True
mla


In [6]:
gpu_blackwell = sb.get_gpu_info('./device/gpu_info.csv',print_console=True ) 

| gpu_type      |   sm |   comm_sm |   fp16 |   fp8 |   fp4 |   mem |   mem_bw |   nvlink_bw |   pcie_bw |   gpu_per_node |
|:--------------|-----:|----------:|-------:|------:|------:|------:|---------:|------------:|----------:|---------------:|
| DGX-B300      |  160 |        20 |   3375 |  7500 | 15000 |   288 |     8000 |         900 |       100 |              8 |
| DGX-B200      |  160 |        20 |   2250 |  4500 |  9000 |   180 |     8000 |         900 |       100 |              8 |
| GB200-NVL72   |  160 |        20 |   2500 |  5000 | 10000 |   192 |     8000 |         900 |       100 |             72 |
| GB300-NVL72   |  160 |        20 |   3750 |  7500 | 15000 |   288 |     8000 |         900 |       100 |             72 |
| Rubin-NVL144  |  110 |        12 |   6400 | 12800 | 25600 |   144 |     6500 |         900 |       100 |            144 |
| RubinU-NVL576 |  110 |        12 |   6500 | 13000 | 26000 |   256 |     8000 |        1350 |       100 |            576 |
| H200  

In [7]:
seq_len = 4383
kv_cache_rate = 0.563
decode_len = 1210
bs_list =[ 16, 32, 64, 128, 256, 512]
eplist = [ 8 , 16, 36, 72, 144, 320]

In [8]:
detail,summary = sb.prefill_time(args,gpu_blackwell,seq_len, kv_cache_rate, tp=4, dp=8)

In [9]:
detail

GPU,Layers,DGX-B300,DGX-B200,GB200-NVL72,GB300-NVL72,Rubin-NVL144,RubinU-NVL576,H200,H800,H20,H20-3E,MI300X,MI308X
MLA,3.0,1.534450,2.310493,2.079444,1.386296,0.797778,0.785504,5.848370,5.857117,37.806369,37.797622,4.174700,28.484686
DenseMLP,3.0,0.367029,0.574656,0.522749,0.367029,0.247642,0.232057,2.527005,2.545534,15.951653,15.933123,1.818876,12.018666
TP_MLA,58.0,0.526944,0.720955,0.663193,0.489906,0.342776,0.313597,1.683756,1.881772,9.673256,9.671069,1.266038,7.343534
Shared Expert,58.0,0.014828,0.020595,0.019153,0.014828,0.012580,0.011078,0.077915,0.079974,0.452366,0.450307,0.057517,0.340844
Combine,58.0,0.578740,0.578740,0.204216,0.204216,0.205440,0.154034,1.107479,1.107479,1.107479,1.107479,1.107479,1.107479
Overlap1,58.0,0.036968,-0.162811,-0.478130,-0.300518,-0.149917,-0.170641,-0.654192,-0.854267,-9.018143,-9.013897,-0.216075,-6.576899
Routed Expert,58.0,0.118621,0.164760,0.153225,0.118621,0.100642,0.088627,0.623321,0.639792,3.618929,3.602459,0.460132,2.726753
Dispatch,58.0,0.182185,0.182185,0.088554,0.088554,0.088860,0.076009,0.314370,0.314370,0.314370,0.314370,0.314370,0.314370
Overlap2,58.0,0.063564,0.017425,-0.064671,-0.030067,-0.011783,-0.012618,-0.308951,-0.325422,-3.304560,-3.288089,-0.145763,-2.412383


In [10]:
summary

GPU,DGX-B300,DGX-B200,GB200-NVL72,GB300-NVL72,Rubin-NVL144,RubinU-NVL576,H200,H800,H20,H20-3E,MI300X,MI308X
Compute,44.007206,61.221441,56.269710,41.414515,29.584204,27.024238,163.455698,176.097174,958.458037,957.174671,121.434578,725.355665
Comm,44.133625,44.133625,16.980641,16.980641,17.069376,13.342496,82.467251,82.467251,82.467251,82.467251,82.467251,82.467251
Sum,49.838061,62.232083,56.269710,41.414515,29.584204,27.024238,163.455698,176.097174,958.458037,957.174671,121.434578,725.355665


In [26]:
detail,summary = sb.prefill_time_pp(args,gpu_blackwell,seq_len, kv_cache_rate, tp=4, dp=8,bs=32, pp=4, micro_batch = 1)

TypeError: prefill_time_pp() got an unexpected keyword argument 'micro_batch'

In [ ]:
summary

In [ ]:
tp=4
_ , ttft_sum = sb.prefill_time(args,gpu_blackwell,seq_len, kv_cache_rate, tp=tp, dp=8, print_console=False)
print(ttft_sum.apply(lambda x: seq_len/tp * (1000/ x)).loc['Sum'].to_markdown(floatfmt=".1f"))

In [ ]:
args = sb.ModelArgs.load_from_csv("modelArgs.csv","llama3")
c = sb.Config()
c.bs_list=[16,32,64,128,256,512,1024,2048]
gpu_blackwell_decode = sb.get_gpu_info('./device/gpu_info.csv', 
                                    decoding_mode=True, print_console=True) 

In [ ]:
# test llama3
detail = sb.decode_time(
        args,                     # llama-3 ModelArgs   (is_moe=False)
        gpu_blackwell_decode,                 # your GPU list
        c.bs_list,           # batch-size sweep
        seq_len,
        decode_len,
        gemm_group_per_device=0,  # ← ignored when is_moe=False
        device_num=8,             # ← ignored when is_moe=False
        fp8_combine=False,
        tps_limit=0,
        print_console=True) 


In [ ]:
result = sb.decode_time_pp(
        args,                     # llama-3 ModelArgs   (is_moe=False)
        gpu_blackwell_decode,                
        c.bs_list,           # batch-size sweep
        seq_len,
        decode_len,
        gemm_group_per_device=0,  
        device_num=8,   
        pp=4,                   # pipeline parallelism          
        fp8_combine=False,
        tps_limit=0,
        print_console=True) 

In [ ]:
dfs_o = detail.groupby(['GPU','BatchSize'],as_index=False).apply(lambda t: t[t.Total==t.Total.max()]).sort_values(['Total'],ascending=False).reset_index(drop=True)

dfs_o.style.bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.gpu_category_color,props=sb.gpu_category_idx(gpu_blackwell_decode),subset=['GPU'])\
      .format(precision=3) 

In [ ]:
sb.df_filter(dfs_o,'Rubin-NVL144',0).style\
      .bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.color_positive_red, subset=['Delta'])\
      .background_gradient(subset=['Comm_Impact'],cmap=sb.cm)\
      .format(precision=3) 

In [ ]:
args = sb.ModelArgs.load_from_csv("modelArgs.csv","deepseek-v3")
c = sb.Config()
c.bs_list=[16,32,64,128,256,512,1024,2048]
gpu_blackwell_decode = sb.get_gpu_info('./device/gpu_info.csv', 
                                    decoding_mode=True, print_console=True) 

In [ ]:
dfs = sb.decode_time_with_ep_list(args,gpu_blackwell_decode,c,print_console=False,fp8_combine=True,tps_limit=0)

In [ ]:
dfs_o = dfs.groupby(['GPU','BatchSize'],as_index=False).apply(lambda t: t[t.Total==t.Total.max()]).sort_values(['Total'],ascending=False).reset_index(drop=True)
dfs_o.style.bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.gpu_category_color,props=sb.gpu_category_idx(gpu_blackwell_decode),subset=['GPU'])\
      .applymap(sb.color_positive_red, subset=['Delta'])\
      .background_gradient(subset=['Comm_Impact'],cmap=sb.cm)\
      .format(precision=3) 


In [ ]:
sb.df_filter(dfs,'Rubin-NVL144',0).style\
      .bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.color_positive_red, subset=['Delta'])\
      .background_gradient(subset=['Comm_Impact'],cmap=sb.cm)\
      .format(precision=3) 